In [1]:
# Homework 3: Machine Learning for Classification

In [2]:
## 1. Load and inspect the lead-scoring dataset

In [3]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)

print("Dataset shape:", df.shape)


Dataset shape: (5000, 9)


In [4]:
display(df.head())

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
print ("column data types:")
print (df.dtypes)

column data types:
lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object


In [6]:
# count missing values in each column 
print ("\n missing value per column:")
print (df.isna().sum())


 missing value per column:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [7]:
print(df["converted"].value_counts())

converted
1    2883
0    2117
Name: count, dtype: int64


In [8]:
categorical = ["lead_source", "industry", "employment_status", "location",] 
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score",]
df[categorical] =df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print(df.isna().sum())

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [9]:
# Q1: What is the most frequent industry?

In [10]:
# show no of leads in each industry 
print (df["industry"].value_counts()) 

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


In [11]:
# show the most frequent indutry 
print("\n Mode:", df["industry"].mode().iloc[0])


 Mode: technology


In [12]:
## Q2: correlation between numerical features

In [13]:
# Calculate correlations between our four numerical features

In [14]:
correlation_matrix = df[numerical].corr()
display(correlation_matrix)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [15]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for feature_1, feature_2 in pairs:
    score = correlation_matrix.loc[feature_1, feature_2]
    print(f"{feature_1} and {feature_2}: {score:.4f}")

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [16]:
print("Total rows:", len(df))

Total rows: 5000


In [17]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [18]:
from sklearn.model_selection import train_test_split

In [19]:
# Reserve 20% of all rows for testing
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

# Reserve 25% of the remaining rows for validation
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

print("Training rows:", len(df_train))
print("Validation rows:", len(df_val))
print("Test rows:", len(df_test))

Training rows: 3000
Validation rows: 1000
Test rows: 1000


In [20]:
# Save the answers separately and remove them from the feature tables
y_train = df_train["converted"].to_numpy()
y_val = df_val["converted"].to_numpy()
y_test = df_test["converted"].to_numpy()

df_train = df_train.drop(columns=["converted"])
df_val = df_val.drop(columns=["converted"])
df_test = df_test.drop(columns=["converted"])

print("Training features:", df_train.shape)
print("Training targets:", y_train.shape)
print("Feature columns:", df_train.columns.tolist())

Training features: (3000, 8)
Training targets: (3000,)
Feature columns: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [21]:
##  Q3: Which categorical feature has the biggest mutual information score?


In [22]:
from sklearn.metrics import mutual_info_score

# Compare each categorical feature with the training target
mi_scores = {}

for column in categorical:
    score = mutual_info_score(df_train[column], y_train)
    mi_scores[column] = score

# Display largest scores first
mi_results = pd.Series(mi_scores).sort_values(ascending=False)

for column, score in mi_results.items():
    print(f"{column}: {round(score, 2)}")

lead_source: 0.03
employment_status: 0.02
industry: 0.0
location: 0.0


In [23]:
## Q4, starting with one-hot encoding before fitting the model

In [24]:
from sklearn.feature_extraction import DictVectorizer

# Represent each lead as a dictionary of feature names and values
train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

# Learn the encoding from training data only
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

# Apply that same encoding to validation data
X_val = dv.transform(val_dicts)

print("Encoded training shape:", X_train.shape)
print("Encoded validation shape:", X_val.shape)
print("Encoded feature names:")
print(dv.get_feature_names_out())

Encoded training shape: (3000, 28)
Encoded validation shape: (1000, 28)
Encoded feature names:
['annual_income' 'employment_status=NA' 'employment_status=employed'
 'employment_status=self_employed' 'employment_status=student'
 'employment_status=unemployed' 'industry=NA' 'industry=education'
 'industry=finance' 'industry=healthcare' 'industry=manufacturing'
 'industry=retail' 'industry=technology' 'interaction_count' 'lead_score'
 'lead_source=NA' 'lead_source=events' 'lead_source=organic_search'
 'lead_source=paid_ads' 'lead_source=referral' 'lead_source=social_media'
 'location=NA' 'location=africa' 'location=asia' 'location=europe'
 'location=north_america' 'location=south_america'
 'number_of_courses_viewed']


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create the model with the exact parameters
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42,
)

# Learn from training features and their correct answers
model.fit(X_train, y_train)

# Predict conversion labels for validation leads
y_pred = model.predict(X_val)

# Compare predictions with the actual validation outcomes
baseline_accuracy = accuracy_score(y_val, y_pred)

print("Validation accuracy:", baseline_accuracy)
print("Rounded accuracy:", round(baseline_accuracy, 2))

Validation accuracy: 0.645
Rounded accuracy: 0.65


In [26]:
## How does validation accuracy change when we remove one feature and retrain?

In [27]:
features = categorical + numerical
elimination_results = []

for removed_feature in features:
    remaining_features = [
        feature for feature in features
        if feature != removed_feature
    ]

    # Encode the remaining features
    dv_reduced = DictVectorizer(sparse=False)

    X_train_reduced = dv_reduced.fit_transform(
        df_train[remaining_features].to_dict(orient="records")
    )
    X_val_reduced = dv_reduced.transform(
        df_val[remaining_features].to_dict(orient="records")
    )

    # Train a fresh model with the same parameters as Q4
    model_reduced = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42,
    )

    model_reduced.fit(X_train_reduced, y_train)
    predictions = model_reduced.predict(X_val_reduced)

    accuracy_without = accuracy_score(y_val, predictions)
    difference = baseline_accuracy - accuracy_without

    elimination_results.append({
        "removed_feature": removed_feature,
        "accuracy_without": accuracy_without,
        "difference": difference,
    })

results = pd.DataFrame(elimination_results)
display(results.sort_values("difference"))

,removed_feature,accuracy_without,difference
4,annual_income,0.724,-0.079
1,industry,0.645,0.000
2,employment_status,0.645,0.000
3,location,0.645,0.000
7,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


In [28]:
## Q6: regularization.

In [29]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]
regularization_results = []

for C in C_values:
    model_regularized = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42,
    )

    model_regularized.fit(X_train, y_train)
    predictions = model_regularized.predict(X_val)

    accuracy = accuracy_score(y_val, predictions)
    rounded_accuracy = round(accuracy, 3)

    regularization_results.append({
        "C": C,
        "accuracy": accuracy,
        "rounded_accuracy": rounded_accuracy,
    })

    print(f"C={C:g}: accuracy={rounded_accuracy:.3f}")

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645
